In [0]:
import zipfile, io, os, glob, shutil

RAW = "/Volumes/sepa/landing/raw"


def descomprimir(fecha):
    """Descomprime el zip de un dia en landing/raw/csv/fecha=.../comercio=.../"""
    zip_path = f"{RAW}/zips/sepa_{fecha}.zip"
    destino = f"{RAW}/csv/fecha={fecha}"
    ok, vacios, errores = [], [], []
    with zipfile.ZipFile(zip_path) as outer:
        for nombre in outer.namelist():
            if not nombre.endswith(".zip"):
                continue
            comercio = os.path.basename(nombre).split("_")[2]
            datos = outer.read(nombre)
            if len(datos) == 0:
                vacios.append(comercio)
                continue
            try:
                carpeta = f"{destino}/comercio={comercio}"
                os.makedirs(carpeta, exist_ok=True)
                with zipfile.ZipFile(io.BytesIO(datos)) as inner:
                    inner.extractall(carpeta)
                ok.append(comercio)
            except zipfile.BadZipFile:
                errores.append(comercio)
    return ok, vacios, errores


def reparar_csv(ruta):
    """Une las filas que vienen partidas en varias lineas.
    Devuelve cuantas filas tuvo que reparar."""
    temporal = ruta + ".tmp"
    reparadas = 0
    with open(ruta, encoding="utf-8", newline="") as entrada, \
         open(temporal, "w", encoding="utf-8", newline="") as salida:
        encabezado = entrada.readline().rstrip("\r\n")
        salida.write(encabezado + "\n")
        separadores = encabezado.count("|")
        buffer, partes = "", 0
        for linea in entrada:
            linea = linea.rstrip("\r\n")
            if buffer == "" and "|" not in linea:
                salida.write(linea + "\n")
                continue
            buffer += linea
            partes += 1
            if buffer.count("|") >= separadores:
                salida.write(buffer + "\n")
                if partes > 1:
                    reparadas += 1
                buffer, partes = "", 0
        if buffer:
            salida.write(buffer + "\n")
    os.replace(temporal, ruta)
    return reparadas

In [0]:
# 1. Corregir zips subidos con doble extension (.zip.zip)
for ruta in glob.glob(f"{RAW}/zips/*.zip.zip"):
    os.rename(ruta, ruta[:-4])

# 2. Detectar que dias faltan procesar
fechas_zip = sorted(os.path.basename(z)[5:15] for z in glob.glob(f"{RAW}/zips/sepa_*.zip"))
pendientes = [f for f in fechas_zip
              if not os.path.exists(f"{RAW}/csv/fecha={f}/_OK")]
print(f"Zips disponibles: {fechas_zip}")
print(f"Pendientes: {pendientes}")

# 3. Procesar cada dia pendiente
for fecha in pendientes:
    carpeta = f"{RAW}/csv/fecha={fecha}"
    if os.path.exists(carpeta):
        shutil.rmtree(carpeta)
    ok, vacios, errores = descomprimir(fecha)
    reparadas = sum(reparar_csv(r) for r in glob.glob(f"{carpeta}/*/*.csv"))
    open(f"{carpeta}/_OK", "w").close()
    print(f"{fecha}: {len(ok)} comercios OK | vacios: {vacios} | errores: {errores} | filas reparadas: {reparadas}")

print("Listo")